# W2M5 - Team Activity
- 카카오톡 앱의 구글 플레이 리뷰 분석을 통해 업데이트 이후의 반응 변화 제공

## Import Module

In [ ]:
!pip install -r requirements.txt

In [ ]:
import csv
import os
import re
import time
import math
from datetime import datetime
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
from kiwipiepy import Kiwi
from wordcloud import WordCloud

# macOS에서 ChromDriver를 활용해 Selenium을 활용하기 위해
# Homebrew를 통해 설치 (brew install chormedriver)

# 보안 메시지가 뜨는 것을 지우는 터미널 명령
# xattr -d com.apple.quarantine $(which chromedriver)
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

## Config

In [ ]:
# Play Store - KaKaotalk 앱 url
URL = "https://play.google.com/store/apps/details?id=com.kakao.talk&hl=ko"

# 리뷰 수집 구간 설정
# 개편_이전: 2025년 9월 대규모 업데이트 전 리뷰
# 개편_직후_반발: 2025년 9월 대규모 업데이트 후 ~ 원상 복구 전 리뷰
# 원복_완료_후: 원상 복구 후 리뷰
TIMELINE_SECTIONS = [
    {"label": "개편_이전", "start": datetime(2023, 1, 1), "end": datetime(2025, 9, 22, 23, 59, 59)},
    {"label": "개편_직후_반발", "start": datetime(2025, 9, 23), "end": datetime(2025, 12, 31, 23, 59, 59)},
    {"label": "원복_완료_후", "start": datetime(2026, 1, 1), "end": None},
]
TARGET_PER_SECTION = 500 # 구간별로 최대 500개씩 리뷰 웹스크래핑

# 저장할 파일 이름
TABLET_CSV = "kakaotalk_reviews_tablet.csv"
PHONE_CSV = "kakaotalk_reviews_phone.csv"
MERGED_CSV = "kakaotalk_reviews_merged.csv"
WORDCLOUD_PNG = "kakaotalk_reviews_wordcloud.png"

FONT_PATH = "/System/Library/Fonts/AppleSDGothicNeo.ttc"  # 한글 폰트 (macOS 기준)

# 워드클라우드에서 제외할, 분석적으로 의미 없는 명사들
EXCLUDED_NOUNS = {"카카오톡", "카톡", "업데이트", "업뎃", "이번", "사용", "기능", "때"}
# 기본 사전에 없어서 두 명사로 쪼개지는 복합명사
CUSTOM_NOUNS = ["오픈채팅", "친구탭", "숏폼"]

# 웹스크래핑을 Skip 할 것인지 (기본값 False)
SKIP_TABLET = False
SKIP_PHONE = False

## Extract

In [ ]:
# Utils Function
# 리뷰 -> 구간에 배정
def assign_section(review_date):
    for section in TIMELINE_SECTIONS:
        if review_date < section["start"]:
            continue
        if section["end"] is not None and review_date > section["end"]:
            continue
        return section["label"]
    return None


# 클릭을 통해 웹페이지가 해당 창을 인식하게끔 만드는 함수
def _real_click(driver, el):
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", el)
    ActionChains(driver).move_to_element(el).pause(0.15).click().perform()


# 별점 추출
def _extract_rating(block):
    try:
        star_el = block.find_element(By.CSS_SELECTOR, "[aria-label*='별표'], div[role='img']")
        label = star_el.get_attribute("aria-label") or ""
        m = re.search(r"(\d+(?:\.\d+)?)개를?\s*받았습니다", label)
        return float(m.group(1)) if m else None
    except Exception: # 웹 스크래핑의 오류가 있다면
        return None


# device별로 리뷰 선택
def scrape_reviews(device, output_csv, target_per_section=TARGET_PER_SECTION):
    options = webdriver.ChromeOptions()
    options.add_argument("--blink-settings=imagesEnabled=false")
    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 15)
    driver.get(URL)

    seen = set()
    section_counts = {s["label"]: 0 for s in TIMELINE_SECTIONS}
    file_exists = os.path.exists(output_csv)

    if file_exists:
        with open(output_csv, "r", encoding="utf-8-sig", newline="") as f:
            for row in csv.DictReader(f):
                seen.add((row["date"], row["content"]))
                if row["period"] in section_counts:
                    section_counts[row["period"]] += 1

    csv_file = open(output_csv, "a", encoding="utf-8-sig", newline="")
    writer = csv.DictWriter(csv_file, fieldnames=["date", "rating", "period", "content"])
    if not file_exists:
        writer.writeheader()

    try:
        if device == "태블릿":
            tablet_tab = wait.until(
                EC.element_to_be_clickable((By.XPATH, "//div[@role='button' and @aria-label='태블릿']"))
            )
            _real_click(driver, tablet_tab)
            time.sleep(1)
            print(f"기기: {device}로 전환")

        review_button = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, "//button[contains(., '리뷰 모두 보기') or contains(., '모든 리뷰')]")
            )
        )
        _real_click(driver, review_button)

        dialog = wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, "div[role='dialog']")))

        try:
            sort_trigger = WebDriverWait(dialog, 10).until(
                EC.element_to_be_clickable(
                    (
                        By.XPATH,
                        ".//div[@role='button']"
                        "[contains(@aria-label, '관련성') or contains(@aria-label, '정렬') " # 관련성으로 정렬
                        "or contains(@id, 'sortBy')]",
                    )
                )
            )
            _real_click(driver, sort_trigger)

            relevance_option = wait.until(
                EC.element_to_be_clickable(
                    (
                        By.XPATH,
                        "//*[@role='menuitemradio']"
                        "[@aria-label='관련성순' or contains(@aria-label, '관련성')]",
                    )
                )
            )
            _real_click(driver, relevance_option)
            time.sleep(1)
            print("정렬: 관련성순 적용 완료")
        except Exception as e:
            print("정렬 옵션을 찾지 못했습니다 (이미 관련성순이 기본값일 수 있음):", e)

        #실제 스크롤 컨테이너 찾기
        first_review = dialog.find_element(By.CSS_SELECTOR, "div.RHo1pe")
        scroll_container = driver.execute_script(
            """
            let node = arguments[0];
            while (node) {
                const style = window.getComputedStyle(node);
                if ((style.overflowY === 'auto' || style.overflowY === 'scroll')
                    && node.scrollHeight > node.clientHeight) {
                    return node;
                }
                node = node.parentElement;
            }
            return null;
            """,
            first_review,
        )
        if scroll_container is None:
            raise RuntimeError("스크롤 가능한 컨테이너를 찾지 못했습니다.")

        processed_count = 0

        def collect_new_blocks():
            nonlocal processed_count
            added = 0
            review_blocks = scroll_container.find_elements(By.CSS_SELECTOR, "div.RHo1pe")
            new_blocks = review_blocks[processed_count:]
            processed_count = len(review_blocks)

            for block in new_blocks:
                try:
                    date_text = block.find_element(By.CSS_SELECTOR, "span.bp9Aid").text
                    review_date = datetime.strptime(date_text, "%Y년 %m월 %d일")
                    content = block.find_element(By.CSS_SELECTOR, "div.h3YV2d").text

                    key = (review_date.strftime("%Y-%m-%d"), content)
                    if key in seen:
                        continue
                    seen.add(key)

                    section_label = assign_section(review_date)
                    if section_label is None:
                        continue
                    if section_counts[section_label] >= target_per_section:
                        continue

                    writer.writerow({
                        "date": key[0],
                        "rating": _extract_rating(block),
                        "period": section_label,
                        "content": content,
                    })
                    csv_file.flush()

                    section_counts[section_label] += 1
                    added += 1
                except Exception:
                    continue
            return added

    # 메인 크롤링 코드 부분
        iteration = 0
        no_progress_count = 0
        max_no_progress = 6

        while True:
            iteration += 1
            before_seen = len(seen)

            driver.execute_script(
                "arguments[0].scrollTop += arguments[0].clientHeight * 2;", scroll_container
            )
            time.sleep(1)

            added = collect_new_blocks()
            after_seen = len(seen)

            counts_text = ", ".join(f"{s['label']}={section_counts[s['label']]}" for s in TIMELINE_SECTIONS)
            print(f"[{device} iter {iteration}] 신규 저장 {added}개 | {counts_text}")

            if all(section_counts[s["label"]] >= target_per_section for s in TIMELINE_SECTIONS):
                print(f"모든 구간이 {target_per_section}개 이상 수집됨 → 종료")
                break

            if after_seen == before_seen:
                no_progress_count += 1
                if no_progress_count >= max_no_progress:
                    print("더 이상 로딩되는 리뷰 없음 → 종료")
                    break
            else:
                no_progress_count = 0

    finally:
        csv_file.close()
        driver.quit()

    print(f"[{device}] 완료. 저장 파일: {output_csv}")
    return pd.read_csv(output_csv, encoding="utf-8-sig")

In [ ]:
# 메인 Extract 로직
def extract(skip_tablet=False, skip_phone=False):
    if skip_tablet:
        print(f"태블릿 스크래핑 건너뜀 -> 기존 파일 로드: {TABLET_CSV}")
        tablet_df = pd.read_csv(TABLET_CSV, encoding="utf-8-sig")
    else:
        tablet_df = scrape_reviews("태블릿", TABLET_CSV)

    if skip_phone:
        print(f"폰 스크래핑 건너뜀 -> 기존 파일 로드: {PHONE_CSV}")
        phone_df = pd.read_csv(PHONE_CSV, encoding="utf-8-sig")
    else:
        phone_df = scrape_reviews("폰", PHONE_CSV)

    return tablet_df, phone_df

## Transform

In [ ]:
# 명사 추출 함수
def _extract_nouns(text, kiwi):
    NOUN_TAGS = {"NNG", "NNP"}
    nouns = [t.form for t in kiwi.tokenize(text) if t.tag in NOUN_TAGS]
    nouns = [n for n in nouns if n not in EXCLUDED_NOUNS]
    return " ".join(nouns)

In [ ]:
# 메인 transform 함수
def transform(tablet_df, phone_df):
    df = pd.concat([tablet_df, phone_df], ignore_index=True)

    # 평점 4점 이상 긍정, 2점 이하 부정, 3점은 분석에서 제외
    df["sentiment"] = None
    df.loc[df["rating"] >= 4, "sentiment"] = "positive"
    df.loc[df["rating"] <= 2, "sentiment"] = "negative"
    df = df.dropna(subset=["sentiment"]).reset_index(drop=True)

    # 워드클라우드용 명사 추출
    kiwi = Kiwi()
    for word in CUSTOM_NOUNS:
        kiwi.add_user_word(word, "NNG")

    df["nouns"] = df["content"].astype(str).apply(lambda text: _extract_nouns(text, kiwi))

    print(df.groupby(["period", "sentiment"]).size())
    return df

## Load

In [ ]:
# 워드클라우드 생성 함수
def _make_wordcloud(text):
    return WordCloud(font_path=FONT_PATH, width=800, height=400, 
                     background_color="white", max_words=100).generate(text)

In [ ]:
def load(df, merged_csv=MERGED_CSV):
    plt.rcParams["font.family"] = "AppleGothic"
    plt.rcParams["axes.unicode_minus"] = False

    df.to_csv(merged_csv, index=False, encoding="utf-8-sig")
    print(f"병합 CSV 저장 완료: {merged_csv}")

## Main

In [ ]:
tablet_df, phone_df = extract(skip_tablet=SKIP_TABLET, skip_phone=SKIP_PHONE)
df = transform(tablet_df, phone_df)
load(df)

## Make WordCloud

In [ ]:
# 그룹별(기간, 감정)로 다른 그룹 대비 유독 많이 쓰인 단어를 뽑기 위한 log-odds 계산
# transform 단계에서 이미 명사만 추출된 df["nouns"]를 그대로 재사용
MIN_LEN = 2  # 한 글자 명사는 배제
STOPWORDS = {
    "때문", "정도", "이거", "저거", "그거", "누구", "무엇", "얘",
    "제발", "그냥", "진짜", "정말", "요즘", "지금", "생각", "사람", "이건", "하나",
}


def _nouns_counter(nouns_series):
    counter = Counter()
    for nouns_str in nouns_series.astype(str):
        for w in nouns_str.split():
            if len(w) >= MIN_LEN and w not in STOPWORDS:
                counter[w] += 1
    return counter


def log_odds_zscores(counter_i, counter_rest, alpha0=200, z_min=1.96, top=100):
    n_i, n_j = sum(counter_i.values()), sum(counter_rest.values())
    total = counter_i + counter_rest
    n = n_i + n_j
    scores = {}
    for w, y_all in total.items():
        a_w = alpha0 * y_all / n
        y_i, y_j = counter_i.get(w, 0), counter_rest.get(w, 0)
        d = (math.log((y_i + a_w) / (n_i + alpha0 - y_i - a_w))
            - math.log((y_j + a_w) / (n_j + alpha0 - y_j - a_w)))
        z = d / math.sqrt(1 / (y_i + a_w) + 1 / (y_j + a_w))
        if z > z_min:
            scores[w] = z
    return dict(sorted(scores.items(), key=lambda x: -x[1])[:top])


def _make_wordcloud_from_scores(scores):
    return WordCloud(font_path=FONT_PATH, width=800, height=400,
                      background_color="white", max_words=100).generate_from_frequencies(scores)

In [ ]:
periods = [s["label"] for s in TIMELINE_SECTIONS]
sentiments = ["positive", "negative"]

fig, axes = plt.subplots(len(periods), len(sentiments), figsize=(16, 18))

for i, period in enumerate(periods):
    for j, sentiment in enumerate(sentiments):
        is_group = (df["period"] == period) & (df["sentiment"] == sentiment)
        counter_i = _nouns_counter(df.loc[is_group, "nouns"])
        counter_rest = _nouns_counter(df.loc[~is_group, "nouns"])
        scores = log_odds_zscores(counter_i, counter_rest)

        ax = axes[i, j]
        if scores:
            ax.imshow(_make_wordcloud_from_scores(scores))
        ax.axis("off")
        ax.set_title(f"{period} / {sentiment} (n={is_group.sum()})")

plt.tight_layout()
plt.savefig(WORDCLOUD_PNG, dpi=150)
print(f"워드클라우드 이미지 저장 완료: {WORDCLOUD_PNG}")
plt.show()